# 01 — Data Pipeline

Runs the full data pipeline for **AirfRANS** and **CFDBench**:
1. Mount Drive + clone/pull repo (self-contained — safe to re-run)
2. Install dependencies
3. Download AirfRANS (scarce split = 200 cases, ~3 GB)
4. Download CFDBench (4 problems)
5. Convert both datasets → unified HDF5 schema
6. Embed dimensionless numbers per case
7. Validate output + print summary stats

**CPU is fine for this notebook.**  
**Estimated runtime (Colab CPU):** ~30 min download + ~20 min convert for scarce split.

## 0. Mount Drive + clone/pull repo

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# ── Mount Drive ───────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
    BASE_DIR = '/content/drive/MyDrive/cfd-ald-app'
    print('Drive mounted.')
except ImportError:
    ON_COLAB = False
    BASE_DIR = os.path.expanduser('~/cfd-ald-app-data')
    print(f'Not on Colab — using local BASE_DIR: {BASE_DIR}')

# ── Clone / pull repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Repo exists — pulling latest...')
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date.')
    if r.returncode != 0:
        print('git pull warning:', r.stderr.strip())
else:
    print(f'Cloning {REPO_URL} → {REPO_DIR} ...')
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode == 0:
        print('Clone successful.')
    else:
        raise RuntimeError(f'git clone failed:\n{r.stderr}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Data paths ────────────────────────────────────────────────────────────
RAW_DIR       = Path(BASE_DIR) / 'data' / 'raw'
PROCESSED_DIR = Path(BASE_DIR) / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nRepo      → {REPO_DIR}')
print(f'Raw data  → {RAW_DIR}')
print(f'Processed → {PROCESSED_DIR}')

## 1. Install dataset-specific packages

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'h5py', 'scipy', 'airfrans', 'fluidfoam', 'tqdm', 'numpy'
], check=True)
print('Dependencies installed.')

## 2. Download AirfRANS

**`scarce` split** = 200 training cases (~3 GB, ~30 min on Colab)  
**`full` split**   = 800 cases (~12 GB, use for final training runs)

Change `AIRFRANS_TASK` below to switch splits.

In [ ]:
AIRFRANS_TASK = 'scarce'   # 'scarce' (200 cases) or 'full' (800 cases)
AIRFRANS_RAW  = str(RAW_DIR / 'airfrans')

manifest = Path(AIRFRANS_RAW) / 'manifest.json'
if manifest.exists():
    existing = json.loads(manifest.read_text())
    print(f'AirfRANS already downloaded: {len(existing)} cases. Skipping.')
else:
    import subprocess
    result = subprocess.run(
        ['python3', f'{REPO_DIR}/data/scripts/download_airfrans.py',
         '--out', AIRFRANS_RAW,
         '--task', AIRFRANS_TASK],
        capture_output=False,   # stream output live
    )
    if result.returncode != 0:
        print('download_airfrans.py failed — check output above.')
    else:
        print('AirfRANS download complete.')

## 3. Download CFDBench

In [ ]:
CFDBENCH_RAW = str(RAW_DIR / 'cfdbench')

manifest = Path(CFDBENCH_RAW) / 'manifest.json'
if manifest.exists():
    existing = json.loads(manifest.read_text())
    print(f'CFDBench already downloaded: {len(existing)} cases. Skipping.')
else:
    import subprocess
    result = subprocess.run(
        ['python3', f'{REPO_DIR}/data/scripts/download_cfdbench.py',
         '--out', CFDBENCH_RAW,
         '--problems', 'cavity', 'tube', 'dam', 'cylinder'],
        capture_output=False,
    )
    if result.returncode != 0:
        print('download_cfdbench.py failed — check output above.')
    else:
        print('CFDBench download complete.')

## 4. Convert AirfRANS → HDF5 (pointcloud schema)

In [ ]:
AIRFRANS_PROC = str(PROCESSED_DIR / 'airfrans')

case_index = Path(AIRFRANS_PROC) / 'case_index.json'
if case_index.exists():
    n = len(json.loads(case_index.read_text()))
    print(f'AirfRANS already converted: {n} HDF5 cases. Skipping.')
else:
    import subprocess
    result = subprocess.run(
        ['python3', f'{REPO_DIR}/data/scripts/convert_to_hdf5.py', 'airfrans',
         '--raw', AIRFRANS_RAW,
         '--out', AIRFRANS_PROC],
        capture_output=False,
    )
    if result.returncode != 0:
        print('convert_to_hdf5.py (airfrans) failed — check output above.')

## 5. Convert CFDBench → HDF5 (grid schema)

In [ ]:
CFDBENCH_PROC = str(PROCESSED_DIR / 'cfdbench')

case_index = Path(CFDBENCH_PROC) / 'case_index.json'
if case_index.exists():
    n = len(json.loads(case_index.read_text()))
    print(f'CFDBench already converted: {n} HDF5 cases. Skipping.')
else:
    import subprocess
    result = subprocess.run(
        ['python3', f'{REPO_DIR}/data/scripts/convert_to_hdf5.py', 'cfdbench',
         '--raw', CFDBENCH_RAW,
         '--out', CFDBENCH_PROC],
        capture_output=False,
    )
    if result.returncode != 0:
        print('convert_to_hdf5.py (cfdbench) failed — check output above.')

## 6. Embed dimensionless numbers

In [ ]:
import subprocess

for dataset, proc_dir in [('airfrans', AIRFRANS_PROC), ('cfdbench', CFDBENCH_PROC)]:
    print(f'Computing dimensionless numbers for {dataset}...')
    result = subprocess.run(
        ['python3', f'{REPO_DIR}/data/scripts/compute_dimensionless.py',
         '--processed', proc_dir,
         '--dataset', dataset],
        capture_output=False,
    )
    if result.returncode != 0:
        print(f'  compute_dimensionless.py ({dataset}) failed — check output above.')
    else:
        print(f'  {dataset} dimensionless numbers OK.')

## 7. Validation + summary stats

In [ ]:
import h5py
import numpy as np
from pathlib import Path

def validate_dataset(proc_dir: str, representation: str, n_sample: int = 3):
    proc = Path(proc_dir)
    h5_files = sorted(proc.rglob('*.h5'))[:n_sample]

    if not h5_files:
        print(f'  ✗  No HDF5 files found in {proc_dir}')
        return

    total = len(list(proc.rglob('*.h5')))
    print(f'\n── {proc.name} ({representation}) — {total} cases ──')

    meta_path = proc / 'metadata.json'
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        print(f'  Fields:        {meta.get("fields", "n/a")}')
        print(f'  Dimensionless: {meta.get("dimensionless", "n/a")}')

    # Spot-check first file
    with h5py.File(h5_files[0], 'r') as h5:
        print(f'  Sample file:   {h5_files[0].name}')
        def show(name, obj):
            if hasattr(obj, 'shape'):
                print(f'    /{name}: shape={obj.shape}  dtype={obj.dtype}')
        h5.visititems(show)

        if 'inputs/global' in h5:
            g = h5['inputs/global'][:]
            g_cols = list(h5['inputs/global'].attrs.get('columns', []))
            if 'Re' in g_cols:
                Re = float(g[g_cols.index('Re')])
                assert Re > 0, f'Re should be positive, got {Re}'
                print(f'  Re check:      {Re:.1f} ✓')

    print(f'  ✓  {proc.name} looks healthy')


validate_dataset(AIRFRANS_PROC, 'pointcloud')
validate_dataset(CFDBENCH_PROC, 'grid')

## 8. Re distribution plots

In [ ]:
import matplotlib.pyplot as plt
import h5py, numpy as np
from pathlib import Path

def collect_global_feature(proc_dir: str, feature: str, max_files: int = 200):
    values = []
    for h5_path in sorted(Path(proc_dir).rglob('*.h5'))[:max_files]:
        try:
            with h5py.File(h5_path, 'r') as h5:
                if 'inputs/global' not in h5:
                    continue
                g_cols = list(h5['inputs/global'].attrs.get('columns', []))
                if feature in g_cols:
                    g = h5['inputs/global'][:]
                    values.append(float(g[g_cols.index(feature)]))
        except Exception:
            pass
    return np.array(values)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

re_airfrans = collect_global_feature(AIRFRANS_PROC, 'Re')
if len(re_airfrans):
    axes[0].hist(re_airfrans, bins=30, color='steelblue', edgecolor='white')
    axes[0].set_title(f'AirfRANS — Re distribution (n={len(re_airfrans)})')
    axes[0].set_xlabel('Re'); axes[0].set_ylabel('Count')
else:
    axes[0].text(0.5, 0.5, 'No data yet', ha='center', va='center')
    axes[0].set_title('AirfRANS — Re distribution')

re_cfdbench = collect_global_feature(CFDBENCH_PROC, 'Re')
if len(re_cfdbench):
    axes[1].hist(re_cfdbench, bins=30, color='darkorange', edgecolor='white')
    axes[1].set_title(f'CFDBench — Re distribution (n={len(re_cfdbench)})')
    axes[1].set_xlabel('Re'); axes[1].set_ylabel('Count')
else:
    axes[1].text(0.5, 0.5, 'No data yet', ha='center', va='center')
    axes[1].set_title('CFDBench — Re distribution')

plt.tight_layout()
plot_path = str(PROCESSED_DIR / 'Re_distributions.png')
plt.savefig(plot_path, dpi=120)
plt.show()
print(f'Plot saved to {plot_path}')

## 9. Save pipeline summary

In [ ]:
summary = {
    'airfrans': {
        'n_cases':  len(list(Path(AIRFRANS_PROC).rglob('*.h5'))),
        'proc_dir': AIRFRANS_PROC,
    },
    'cfdbench': {
        'n_cases':  len(list(Path(CFDBENCH_PROC).rglob('*.h5'))),
        'proc_dir': CFDBENCH_PROC,
    },
}

summary_path = Path(BASE_DIR) / 'data_pipeline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f'\nSummary saved to {summary_path}')
print('\nData pipeline complete ✓  — ready for 02_pretrain_flow.ipynb')